# AgriDiagnose Model V2 — Experiment A on a free Kaggle GPU

This notebook trains the approved unweighted baseline using **TRAIN and VALIDATION only**. It never loads INTERNAL TEST or PlantDoc TEST. Run cells in order. Training remains disabled until you explicitly set `START_TRAINING = True`.

In [ ]:
# 1. Runtime audit and hard GPU gate — run this before changing packages.
import json, os, platform, shutil, subprocess, sys
import tensorflow as tf
import keras
import numpy as np

print('Python:', sys.version)
print('OS:', platform.platform())
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('NumPy:', np.__version__)
print('Built with CUDA:', tf.test.is_built_with_cuda())
print('All devices:', tf.config.list_physical_devices())
GPUS = tf.config.list_physical_devices("GPU")
print('GPUs:', GPUS)
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('nvidia-smi: not found')
if not tf.test.is_built_with_cuda() or not GPUS:
    raise RuntimeError('KAGGLE_GPU_NOT_AVAILABLE: Settings -> Accelerator -> GPU, restart, then rerun this cell.')

## 2. Approved scientific stack
The experiment requires Python 3.11, TensorFlow 2.15.x, Keras 2.15.x, and NumPy 1.26.4. The next cell inspects compatibility before any installation.

In [ ]:
PYTHON_MINOR = sys.version_info[:2]
TF215_PYTHON_COMPATIBLE = (3, 9) <= PYTHON_MINOR < (3, 12)
APPROVED_STACK_EXACT = (
    PYTHON_MINOR == (3, 11)
    and tf.__version__.startswith('2.15.')
    and keras.__version__.startswith('2.15.')
    and np.__version__ == '1.26.4'
)
print({'python_tf215_compatible': TF215_PYTHON_COMPATIBLE, 'approved_stack_exact': APPROVED_STACK_EXACT})
if not TF215_PYTHON_COMPATIBLE:
    raise RuntimeError('KAGGLE_TF215_RUNTIME_INCOMPATIBLE')
if not APPROVED_STACK_EXACT:
    raise RuntimeError('KAGGLE_APPROVED_STACK_REQUIRED: inspect first; use the opt-in install cell only if Python is compatible.')

In [ ]:
# Optional and manual. Run only after the audit says Python is compatible. Restart the session afterward.
INSTALL_APPROVED_STACK = False
if INSTALL_APPROVED_STACK:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'tensorflow==2.15.0', 'keras==2.15.0', 'numpy==1.26.4'], check=True)
    raise RuntimeError('RESTART_KAGGLE_SESSION_NOW, then rerun the runtime and version gates.')
print('No package changes requested.')

## 3. Clone the public repository at the approved revision
The revision is pinned by the repository preparation PR. No GitHub token is required.

In [ ]:
from pathlib import Path
import re, shutil

REPOSITORY_URL = 'https://github.com/ihebjdey2/ai-plant-disease-detection.git'
APPROVED_CODE_REVISION = '1bf9a89113a1ec68ca655065cacfe66bcbe7254e'
PROJECT_ROOT = Path('/kaggle/working/ai-plant-disease-detection')
if not re.fullmatch(r'[0-9a-f]{40}', APPROVED_CODE_REVISION):
    raise RuntimeError('APPROVED_CODE_REVISION_NOT_PINNED')
if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'checkout', '--detach', APPROVED_CODE_REVISION], cwd=PROJECT_ROOT, check=True)
HEAD = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.strip()
assert HEAD == APPROVED_CODE_REVISION
print('Approved repository HEAD:', HEAD)

In [ ]:
# Optional dependency installation after the version audit. The project pins remain authoritative.
INSTALL_PROJECT_DEPENDENCIES = False
if INSTALL_PROJECT_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements-dev.txt')], check=True)
    raise RuntimeError('RESTART_KAGGLE_SESSION_NOW, then rerun both GPU/version gates and repository setup.')
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=False)

In [ ]:
# Import the committed safety helpers and rerun both gates.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from training.kaggle_experiment_a import *
from training.data_pipeline import load_policy, set_experiment_seeds
from training.experiment_a import build_model, compile_phase1, compile_phase2, configure_phase2, parameter_audit
from training.metrics import MacroF1

AUDIT = runtime_audit()
print(json.dumps(AUDIT, indent=2))
require_kaggle_gpu(AUDIT)
require_approved_stack(AUDIT)
GPU_DEVICES = configure_gpu_memory_growth()
DETERMINISM_ENABLED = False
try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISM_ENABLED = True
except (AttributeError, RuntimeError) as exc:
    print('Deterministic operations could not be fully enabled:', type(exc).__name__)
POLICY = load_policy(PROJECT_ROOT / 'training/config/model-v2-training-policy.json')
SEED = set_experiment_seeds(POLICY)
print('GPU memory growth enabled:', GPU_DEVICES, 'seed:', SEED)

## 4. Configure the five private Kaggle TRAIN-source datasets
Attach only the source data needed by TRAIN and VALIDATION. Do not attach INTERNAL TEST or PlantDoc TEST. Edit only the five paths below if your Kaggle dataset slugs differ.

In [ ]:
SOURCE_ROOTS = {
    'historical': Path('/kaggle/input/agridiagnose-historical'),
    'pldd_up': Path('/kaggle/input/agridiagnose-pldd-up'),
    'seasonal_corn': Path('/kaggle/input/agridiagnose-seasonal-corn'),
    'plantdoc_train': Path('/kaggle/input/agridiagnose-plantdoc-train'),
    'banu_deb': Path('/kaggle/input/agridiagnose-banu-deb'),
}
RESOLVED_ROOTS = kaggle_source_roots(SOURCE_ROOTS)
print({name: str(path) for name, path in RESOLVED_ROOTS.items()})

## 5. Exhaustive preflight — still no training
This verifies all 66,219 TRAIN/VALIDATION files, both 39-class coverages, preprocessing, MacroF1, and the INTERNAL TEST manifest hash. It does not open TEST images.

In [ ]:
WORKING_ROOT = Path('/kaggle/working')
CANDIDATE_DIR = WORKING_ROOT / 'models/candidates/agri-diagnose-v2-exp-a'
RESULTS_DIR = WORKING_ROOT / 'agridiagnose-exp-a-results'
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT = run_full_preflight(PROJECT_ROOT, RESOLVED_ROOTS)
write_json(RESULTS_DIR / 'preflight.json', PREFLIGHT)
print(json.dumps(PREFLIGHT, indent=2))
assert PREFLIGHT['train']['resolved'] == 58857 and PREFLIGHT['train']['missing'] == 0 and PREFLIGHT['train']['unreadable'] == 0
assert PREFLIGHT['validation']['resolved'] == 7362 and PREFLIGHT['validation']['missing'] == 0 and PREFLIGHT['validation']['unreadable'] == 0
assert PREFLIGHT['macro_f1']['passed']
assert PREFLIGHT['internal_test_loaded'] is False and PREFLIGHT['plantdoc_test_loaded'] is False

In [ ]:
# Batch size 32 is approved. Change to 16 or 8 only after a real OOM and document it.
BATCH_SIZE = 32
if BATCH_SIZE not in {32, 16, 8}:
    raise ValueError('Allowed batch sizes are 32, 16, or 8.')
TRAIN_DATASET, VALIDATION_DATASET, TRAIN_RECORDS, VALIDATION_RECORDS = build_kaggle_datasets(
    PROJECT_ROOT, RESOLVED_ROOTS, batch_size=BATCH_SIZE
)
print('TRAIN:', len(TRAIN_RECORDS), 'VALIDATION:', len(VALIDATION_RECORDS), 'batch:', BATCH_SIZE)

## 6. Build and audit the fresh model
This loads official ImageNet weights, never the production V1 model. The optional single-batch forward pass performs no optimization.

In [ ]:
MODEL, BACKBONE = build_model(POLICY, weights='imagenet')
compile_phase1(MODEL, POLICY)
PHASE1_AUDIT = {**parameter_audit(MODEL), 'output_shape': list(MODEL.output_shape), 'backbone_trainable': BACKBONE.trainable}
assert MODEL.output_shape == (None, 39) and BACKBONE.trainable is False
print(PHASE1_AUDIT)
SAMPLE_IMAGES, _ = next(iter(TRAIN_DATASET.take(1)))
with tf.device('/GPU:0'):
    SAMPLE_OUTPUT = MODEL(SAMPLE_IMAGES, training=False)
assert SAMPLE_OUTPUT.shape[-1] == 39
print('Single-batch inference:', SAMPLE_OUTPUT.shape, SAMPLE_OUTPUT.device)

## 7. Manual training authorization
Review every audit above first. Existing phase artifacts are detected; exact interrupted-epoch continuation is not claimed. An interrupted phase must be explicitly restarted.

In [ ]:
START_TRAINING = False
RESTART_INTERRUPTED_PHASE = False
print('Existing artifacts:', detect_existing_phase_artifacts(CANDIDATE_DIR))
if not START_TRAINING:
    print('Training remains disabled. Set START_TRAINING = True only after reviewing all gates.')

## 8. Phase 1 — frozen backbone, at most 10 epochs

In [ ]:
if not START_TRAINING:
    raise RuntimeError('TRAINING_DISABLED_BY_USER')
PHASE1_CHECKPOINT = CANDIDATE_DIR / 'phase1-best.keras'
PHASE1_HISTORY = RESULTS_DIR / 'phase1-history.csv'
PHASE1_RUN_MODE = require_fresh_or_explicit_restart(CANDIDATE_DIR, 'phase1', restart_interrupted_phase=RESTART_INTERRUPTED_PHASE, history_path=PHASE1_HISTORY)
PHASE1_CALLBACKS = build_phase_callbacks(POLICY, checkpoint_path=PHASE1_CHECKPOINT, history_path=PHASE1_HISTORY)
PHASE1_FIT = MODEL.fit(
    TRAIN_DATASET, validation_data=VALIDATION_DATASET, epochs=int(POLICY['phase1']['max_epochs']),
    callbacks=PHASE1_CALLBACKS, class_weight=None, verbose=1
)
PHASE1_BEST = best_history_row(PHASE1_HISTORY)
print('Phase 1 best:', PHASE1_BEST, 'run mode:', PHASE1_RUN_MODE)

## 9. Phase 2 — fine-tune from `block_13_expand`, at most 20 epochs

In [ ]:
if not START_TRAINING:
    raise RuntimeError('TRAINING_DISABLED_BY_USER')
PHASE2_HISTORY = RESULTS_DIR / 'phase2-history.csv'
PHASE2_RUN_MODE = require_fresh_or_explicit_restart(CANDIDATE_DIR, 'phase2', restart_interrupted_phase=RESTART_INTERRUPTED_PHASE, history_path=PHASE2_HISTORY)
PHASE2_MODEL = tf.keras.models.load_model(PHASE1_CHECKPOINT, custom_objects={'MacroF1': MacroF1})
PHASE2_BACKBONE = next(layer for layer in PHASE2_MODEL.layers if isinstance(layer, tf.keras.Model) and 'mobilenetv2' in layer.name)
PHASE2_TRAINABILITY = configure_phase2(PHASE2_BACKBONE, POLICY)
compile_phase2(PHASE2_MODEL, POLICY)
assert PHASE2_TRAINABILITY['first_trainable_backbone_layer'] == 'block_13_expand'
assert PHASE2_TRAINABILITY['trainable_backbone_layer_count'] == 25
assert PHASE2_TRAINABILITY['frozen_batch_normalization_count'] == 52
PHASE2_CHECKPOINT = CANDIDATE_DIR / 'phase2-best.keras'
PHASE2_CALLBACKS = build_phase_callbacks(POLICY, checkpoint_path=PHASE2_CHECKPOINT, history_path=PHASE2_HISTORY)
PHASE2_FIT = PHASE2_MODEL.fit(
    TRAIN_DATASET, validation_data=VALIDATION_DATASET, epochs=int(POLICY['phase2']['max_epochs']),
    callbacks=PHASE2_CALLBACKS, class_weight=None, verbose=1
)
PHASE2_BEST = best_history_row(PHASE2_HISTORY)
print('Phase 2 best:', PHASE2_BEST, 'trainability:', PHASE2_TRAINABILITY)

## 10. VALIDATION-only selection and metrics
Both best checkpoints are compared without using either locked TEST set.

In [ ]:
def evaluate_validation_checkpoint(name, checkpoint, best_row, selection_epoch):
    candidate_model = tf.keras.models.load_model(checkpoint, custom_objects={'MacroF1': MacroF1})
    values = candidate_model.evaluate(VALIDATION_DATASET, return_dict=True, verbose=1)
    scores = candidate_model.predict(VALIDATION_DATASET, verbose=1)
    report = validation_report(VALIDATION_RECORDS, scores)
    return {
        'name': name, 'partition': 'VALIDATION', 'checkpoint': str(checkpoint),
        'epoch': int(best_row['epoch']), 'selection_epoch': int(selection_epoch),
        'val_macro_f1': float(best_row['val_macro_f1']),
        'val_loss': float(values['loss']), 'val_accuracy': float(values['accuracy']),
        'macro_recall': report['overall_validation']['macro_recall'], 'report': report,
    }

PHASE1_COMPLETED_EPOCHS = len(PHASE1_FIT.epoch)
PHASE1_RESULT = evaluate_validation_checkpoint('phase1', PHASE1_CHECKPOINT, PHASE1_BEST, PHASE1_BEST['epoch'])
PHASE2_RESULT = evaluate_validation_checkpoint('phase2', PHASE2_CHECKPOINT, PHASE2_BEST, PHASE1_COMPLETED_EPOCHS + PHASE2_BEST['epoch'])
SELECTED = select_candidate([PHASE1_RESULT, PHASE2_RESULT])
print('Selected candidate:', SELECTED['name'], 'epoch:', SELECTED['epoch'])

## 11. Persist the candidate, reports, plots, and one downloadable ZIP

In [ ]:
SELECTED_MODEL_PATH = RESULTS_DIR / 'agri-diagnose-v2-exp-a.keras'
shutil.copy2(Path(SELECTED['checkpoint']), SELECTED_MODEL_PATH)
VALIDATION_METRICS = SELECTED['report']
VALIDATION_METRICS['loss'] = SELECTED['val_loss']
VALIDATION_METRICS['accuracy'] = SELECTED['val_accuracy']
write_json(RESULTS_DIR / 'validation-metrics.json', VALIDATION_METRICS)
save_confusion_artifacts(VALIDATION_METRICS, RESULTS_DIR)
plot_learning_curves(PHASE1_HISTORY, PHASE2_HISTORY, RESULTS_DIR)
ENVIRONMENT = {
    'os_environment': 'Kaggle Notebook Linux', 'python_version': platform.python_version(),
    'tensorflow_version': tf.__version__, 'keras_version': keras.__version__,
    'tensorflow_built_with_cuda': bool(tf.test.is_built_with_cuda()),
    'gpu_detected': bool(tf.config.list_physical_devices('GPU')), 'gpus': AUDIT['gpus'],
    'repository_revision': HEAD, 'repository_runtime_location': '/kaggle/working/ai-plant-disease-detection',
    'seed': SEED, 'batch_size': BATCH_SIZE, 'batch_size_recommendation': BATCH_SIZE,
    'train_resolved': PREFLIGHT['train']['resolved'], 'validation_resolved': PREFLIGHT['validation']['resolved'],
    'internal_test_loaded': False, 'plantdoc_test_loaded': False, 'training_performed': True,
    'tensorflow_op_determinism_enabled': DETERMINISM_ENABLED,
    'full_gpu_determinism_guaranteed': False, 'contains_secrets': False,
}
EXPERIMENT = {
    'experiment': EXPERIMENT_NAME, 'selected_phase': SELECTED['name'], 'selected_epoch': SELECTED['epoch'],
    'selection_epoch': SELECTED['selection_epoch'],
    'candidate_sha256': sha256_file(SELECTED_MODEL_PATH), 'candidate_size_bytes': SELECTED_MODEL_PATH.stat().st_size,
    'train_manifest_sha256': sha256_file(PROJECT_ROOT / 'training/datasets/manifests/dataset-v2-train.csv'),
    'validation_manifest_sha256': sha256_file(PROJECT_ROOT / 'training/datasets/manifests/dataset-v2-validation.csv'),
    'policy_sha256': sha256_file(PROJECT_ROOT / 'training/config/model-v2-training-policy.json'),
    'internal_test_manifest_sha256': PREFLIGHT['internal_test_manifest_sha256'],
    'internal_test_loaded': False, 'plantdoc_test_loaded': False, 'training_performed': True,
    'phase1_run_mode': PHASE1_RUN_MODE, 'phase2_run_mode': PHASE2_RUN_MODE,
}
SUMMARY = {
    'selected_phase': SELECTED['name'], 'selected_epoch': SELECTED['epoch'],
    'validation': VALIDATION_METRICS['overall_validation'],
    'real_world_validation': VALIDATION_METRICS['real_world_validation'],
    'major_confusion_pairs': major_confusion_pairs(VALIDATION_METRICS),
    'test_sets_evaluated': False,
}
write_json(RESULTS_DIR / 'environment.json', ENVIRONMENT)
write_json(RESULTS_DIR / 'experiment.json', EXPERIMENT)
write_json(RESULTS_DIR / 'model-v2-exp-a-summary.json', SUMMARY)
OVERALL = VALIDATION_METRICS['overall_validation']
REAL_WORLD = VALIDATION_METRICS['real_world_validation']
REPORT_LINES = [
    '# Model V2 Experiment A', '',
    f"Selected checkpoint: {SELECTED['name']} phase epoch {SELECTED['epoch']}",
    f"VALIDATION accuracy: {OVERALL['accuracy']:.6f}",
    f"VALIDATION macro F1: {OVERALL['macro_f1']:.6f}",
    f"VALIDATION macro recall: {OVERALL['macro_recall']:.6f}",
    f"VALIDATION weighted F1: {OVERALL['weighted_f1']:.6f}",
    f"REAL_WORLD VALIDATION images/classes: {REAL_WORLD['image_count']}/{REAL_WORLD['supported_class_count']}",
    f"REAL_WORLD VALIDATION macro F1: {REAL_WORLD['macro_f1']:.6f}", '',
    '## Major confusion pairs',
]
REPORT_LINES.extend(
    f"- {row['true_class']} -> {row['predicted_class']}: {row['count']}"
    for row in SUMMARY['major_confusion_pairs']
)
REPORT_LINES.extend(['', 'INTERNAL TEST and PlantDoc TEST were not loaded or evaluated.'])
(RESULTS_DIR / 'model-v2-exp-a-report.md').write_text('\n'.join(REPORT_LINES) + '\n', encoding='utf-8')
ARCHIVE = package_results(RESULTS_DIR, WORKING_ROOT / 'agridiagnose-exp-a-results')
print('Download this file from Kaggle Outputs:', ARCHIVE, 'SHA-256:', sha256_file(ARCHIVE))

## Stop
Download `agridiagnose-exp-a-results.zip` and preserve it unchanged. Do not run Experiment B, INTERNAL TEST, PlantDoc TEST, threshold changes, or production deployment until the results receive human review.